### Datos Stream desde archivos en la nube

1. Leer archivos de la "nube" utilizando API - DataStreamReader
2. Transformar el DataFrame y agregar las siguientes columnas:
    - file path: Ruta del archivo en la nube
    - ingestion date: Current Timestamp
3. Escribir los datos de Stream transformados en Tabla Delta Lake

#### 1. Leer archivos de la "nube" utilizando API - DataStreamReader

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

department_schema = StructType(fields = [
    StructField('department_id', IntegerType()),
    StructField('department_name', StringType()),
    StructField('location_id', IntegerType()),
    StructField('created_timestamp', TimestampType())
])

In [0]:
department_df = (spark.readStream
               .format("json")
               .schema(department_schema)
               .option("maxFilesPerTrigger", 1)
               .option("cleanSource", "archive")
               .option("sourceArchiveDir", "/Volumes/zenviro/raw/operational_data/departments_stream_history/")
               .load("/Volumes/zenviro/raw/operational_data/departments_stream/")
               )

#### 2. Transformar el DataFrame y agregar las siguientes columnas:
- file path: Ruta del archivo en la nube
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

department_transformed_df = (
  department_df.withColumn('file_path', col("_metadata.file_path"))
             .withColumn("ingestion_date", current_timestamp())
)

#### 3. Escribir los datos de Stream transformados en Tabla Delta Lake

In [0]:
streaming_query = ( department_transformed_df.writeStream
                    .format("delta")
                    .option("checkpointLocation", "/Volumes/zenviro/raw/operational_data/departments_stream/_checkpoint_stream")
                    .trigger(availableNow=True)
                    .toTable("zenviro.bronze.departments_stream")
                  )

In [0]:
%sql
SELECT * FROM zenviro.bronze.departments_stream;